# Amazon Nova — picking a tier, and proving the small one is enough

Nova is Amazon's own model family and the clearest size ladder on Bedrock. It is
`bedrock-runtime` only: there is no `bedrock-mantle` path, so Converse is the
API.

| Model | Input | Tier |
|---|---|---|
| `nova-micro` | text | cheapest, text only |
| `nova-lite` | text, image, video | low cost, multimodal |
| `nova-pro` | text, image, video | higher quality, multimodal |
| `nova-premier` | — | **legacy, blocked** (see section 4) |

The interesting question with a ladder is never "which is best" — it is "what is
the cheapest tier that still passes". This notebook answers that empirically
rather than by reputation.

Nova also accepts **video** input on lite and pro. Video is out of scope for this
collection, so the vision cells use images; the same content-block shape applies.


### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `bands_png` | generates a small PNG of colour bands, so vision cells have a known answer |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |
| `control_client` | a boto3 `bedrock` client (model and profile catalogues) |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import sys

sys.path.insert(0, "../_shared")

from bedrock import (
    bands_png,
    converse,
    converse_tool_uses,
    endpoints_for,
    resolve_runtime_id,
    runtime_models,
)

REGION = "us-east-1"
MICRO = "amazon.nova-micro-v1"
LITE = "amazon.nova-lite-v1"
PRO = "amazon.nova-pro-v1"
LADDER = [MICRO, LITE, PRO]

catalogue = runtime_models(REGION)
print(f"{'model':<26} {'input':<22} {'inference types'}")
print("-" * 76)
for model in LADDER:
    entry = catalogue[model]
    print(f"{model:<26} {','.join(sorted(entry['in'])):<22} "
          f"{','.join(sorted(entry['infer']))}")

print()
print("endpoints:", {m.split(".")[-1]: endpoints_for(m, REGION) for m in [MICRO]})
print("=> bedrock-mantle is False: Nova is a runtime-only family.")


model                      input                  inference types
----------------------------------------------------------------------------
amazon.nova-micro-v1       TEXT                   INFERENCE_PROFILE,ON_DEMAND,PROVISIONED
amazon.nova-lite-v1        IMAGE,TEXT,VIDEO       INFERENCE_PROFILE,ON_DEMAND,PROVISIONED
amazon.nova-pro-v1         IMAGE,TEXT,VIDEO       INFERENCE_PROFILE,ON_DEMAND,PROVISIONED



endpoints: {'nova-micro-v1': {'mantle': False, 'runtime': True}}
=> bedrock-mantle is False: Nova is a runtime-only family.


## 1. The same task at three tiers

A single easy prompt tells you nothing — every tier passes. Use a task with a
checkable answer and enough structure that a weaker model can visibly fail.

Below: extract three fields as JSON. The check is mechanical, so "did it pass"
is not a judgement call.


In [2]:
import json as jsonlib

PROMPT = (
    "Extract to JSON with keys name, city, years. "
    "Reply with JSON only, no prose.\n\n"
    "Priya has been an engineer in Singapore for eleven years."
)
EXPECTED = {"name": "Priya", "city": "Singapore", "years": 11}


def grade(raw: str) -> str:
    """Did the model return the three fields with the right values?"""
    text = (raw or "").strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        text = text[4:] if text.startswith("json") else text
    try:
        got = jsonlib.loads(text.strip())
    except Exception:
        return "unparseable"
    hits = sum(
        1
        for key, want in EXPECTED.items()
        if str(got.get(key, "")).lower() == str(want).lower()
    )
    return f"{hits}/3 fields correct"


print(f"{'model':<26} {'tokens':>7}  {'verdict':<22} answer")
print("-" * 92)
for model in LADDER:
    text, response = converse(
        model,
        [{"role": "user", "content": [{"text": PROMPT}]}],
        max_tokens=200,
        temperature=0.0,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} {'-':>7}  ERROR {error[:40]}")
        continue
    total = response.get("usage", {}).get("totalTokens", 0)
    print(f"{model:<26} {total:>7}  {grade(text):<22} "
          f"{text.strip()[:34].replace(chr(10), ' ')}")


model                       tokens  verdict                answer
--------------------------------------------------------------------------------------------


amazon.nova-micro-v1            57  3/3 fields correct     {   "name": "Priya",   "city": "Si


amazon.nova-lite-v1             61  3/3 fields correct     ```json {     "name": "Priya",    


amazon.nova-pro-v1              61  3/3 fields correct     ```json {   "name": "Priya",   "ci


## 2. Vision on lite and pro

`nova-micro` is text-only, so sending it an image is a design error rather than a
quality question. The two multimodal tiers get the same generated image with a
known answer, so "did it look" is verifiable.


In [3]:
GROUND_TRUTH = ["red", "green", "blue"]
png = bands_png([(220, 30, 30), (30, 140, 60), (40, 70, 200)])
QUESTION = "List the colours of the horizontal bands, top to bottom. Three words."


def scored(answer: str) -> str:
    lowered = (answer or "").lower()
    hits = [c for c in GROUND_TRUTH if c in lowered]
    return f"{len(hits)}/3"


print(f"generated {len(png)} bytes of PNG; bands are {', '.join(GROUND_TRUTH)}\n")
for model in (LITE, PRO):
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [
                    {"image": {"format": "png", "source": {"bytes": png}}},
                    {"text": QUESTION},
                ],
            }
        ],
        max_tokens=60,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} ERROR {error[:60]}")
    else:
        print(f"{model:<26} {scored(text)}  {text.strip()[:50]}")

# And the design error, so you recognise it.
text, response = converse(
    MICRO,
    [
        {
            "role": "user",
            "content": [
                {"image": {"format": "png", "source": {"bytes": png}}},
                {"text": QUESTION},
            ],
        }
    ],
    max_tokens=40,
    region=REGION,
)
error = (response.get("error") or {}).get("message")
print(f"\n{MICRO} (text-only) with an image:")
print("   ", error[:130] if error else f"unexpectedly accepted: {text.strip()[:60]}")


generated 308 bytes of PNG; bands are red, green, blue



amazon.nova-lite-v1        3/3  Red, Green, Blue


amazon.nova-pro-v1         3/3  Red, green, blue.



amazon.nova-micro-v1 (text-only) with an image:
    This model doesn't support the image content block that you provided. Update the content block and try again.


## 3. Tool use across the ladder

Tool support is not a given at the cheapest tier, so check it rather than assume.
The assertion here is on the *arguments*, not on whether a call happened — a tool
call with wrong operands still reports `stopReason: tool_use`.


In [4]:
TOOLS = [
    {
        "toolSpec": {
            "name": "convert_currency",
            "description": "Convert an amount between two currencies",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "amount": {"type": "number"},
                        "from": {"type": "string"},
                        "to": {"type": "string"},
                    },
                    "required": ["amount", "from", "to"],
                }
            },
        }
    }
]

print(f"{'model':<26} {'stop':<12} tool call")
print("-" * 78)
for model in LADDER:
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [{"text": "Convert 250 SGD to JPY. Use the tool."}],
            }
        ],
        max_tokens=400,
        tools=TOOLS,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} ERROR {error[:44]}")
        continue
    uses = converse_tool_uses(response)
    if not uses:
        print(f"{model:<26} {response.get('stopReason'):<12} (no tool call)")
        continue
    args = uses[0]["input"]
    ok = (
        str(args.get("amount")) in {"250", "250.0"}
        and str(args.get("from", "")).upper() == "SGD"
        and str(args.get("to", "")).upper() == "JPY"
    )
    print(f"{model:<26} {response.get('stopReason'):<12} "
          f"{args}  {'correct' if ok else 'ARGS WRONG'}")


model                      stop         tool call
------------------------------------------------------------------------------


amazon.nova-micro-v1       tool_use     {'amount': 250, 'from': 'SGD', 'to': 'JPY'}  correct


amazon.nova-lite-v1        tool_use     {'amount': 250, 'from': 'SGD', 'to': 'JPY'}  correct


amazon.nova-pro-v1         tool_use     {'amount': 250, 'from': 'SGD', 'to': 'JPY'}  correct


## 4. `nova-premier` is legacy, and Bedrock tells you so

`nova-premier` is still in the catalogue, which is not the same as being callable.
Bedrock marks superseded models as legacy and refuses new usage. The error is
specific and worth recognising, because the same wording gates several older
models across providers.

This is why "read the catalogue" is not sufficient on its own: the catalogue lists
what exists, not what you are allowed to invoke today.


In [5]:
from bedrock import control_client, runtime_client

PREMIER = "amazon.nova-premier-v1"

# The catalogue is perfectly happy about it.
entry = catalogue.get(PREMIER)
print(f"in catalogue      : {PREMIER in catalogue}")
print(f"inference types   : {sorted(entry['infer']) if entry else '-'}")

# So is the entitlement check.
availability = control_client(REGION).get_foundation_model_availability(
    modelId=f"{PREMIER}:0"
)
print("authorizationStatus:", availability.get("authorizationStatus"))
print("entitlement        :", availability.get("entitlementAvailability"))

# And yet:
print("\nactually calling it:")
try:
    runtime_client(REGION).converse(
        modelId=resolve_runtime_id(PREMIER, REGION),
        messages=[{"role": "user", "content": [{"text": "hi"}]}],
        inferenceConfig={"maxTokens": 12},
    )
    print("    accepted")
except Exception as exc:
    print(f"    {type(exc).__name__}")
    print(f"    {str(exc)[-160:]}")


in catalogue      : True
inference types   : ['INFERENCE_PROFILE']


authorizationStatus: AUTHORIZED
entitlement        : AVAILABLE

actually calling it:


    ResourceNotFoundException
    is Model is marked by provider as Legacy and you have not been actively using the model in the last 30 days. Please upgrade to an active model on Amazon Bedrock


## Takeaways

- **Nova is `bedrock-runtime` only.** No `bedrock-mantle` path, so no bearer token
  and no OpenAI-shaped option.
- **Pick the tier with a graded task, not a vibe.** Section 1 gives a mechanical
  pass mark; if `nova-micro` scores 3/3 on your real task, the higher tiers are
  spend without return.
- **`nova-micro` is text-only.** Sending it an image is a design error, not a
  quality trade-off.
- **Assert tool arguments.** `stopReason: tool_use` only says a call was made, not
  that it was right.
- **Catalogue presence is not permission.** `nova-premier` is listed and shows as
  authorised, and still refuses every call because the provider marked it legacy.
  Sweep your intended model list with one cheap call before you design around it.
